# CART-ECG: Differential Mutation-Based Raindrop Optimized R-Peak-Aware Twisted Convolutional Network for Cardiac Abnormality Detection

**Pipeline implemented in this notebook**

1. Data: MIT-BIH Arrhythmia + Fantasia (PhysioNet via `wfdb`; automatic synthetic fallback when offline so the notebook always runs end-to-end)
2. Preprocessing: resampling -> first-order high-pass IIR -> band-pass -> scaling
3. R-peak detection (Pan-Tompkins style) -> beat segmentation aligned to the R-peak -> RR features
4. **FBSE-WD**: Fourier-Bessel Series Expansion based Wasserstein Descriptor
5. **R-TCN**: R-peak-aware Twisted Convolutional Network (twisted convolution + region-interaction attention + descriptor fusion)
6. **DM-RO**: Differential Mutation-based Raindrop Optimization for hyper-parameter tuning
7. Evaluation: class-wise precision / recall / F1 / sensitivity / specificity / AUC, latency, patient-independent testing, comparison with FADLEC, PathFusion-Net, SKD-AMBRNet, ablations, interpretability

> **Important notes**
> * The twisted convolution, FBSE-WD and DM-RO below are my concrete, runnable formulations of the components named in the abstract. If your paper has specific equations, swap them into the marked functions.
> * FADLEC [25], PathFusion-Net [26] and SKD-AMBRNet [30] are implemented as **simplified proxy architectures** (same design idea, not the authors' code). Replace them with the official/reproduced models for the final paper numbers.
> * Labels follow AAMI classes: **N, S, V, F, Q**. Abnormal = anything other than N.

## 1. Setup

In [ ]:
# !pip install -q wfdb numpy scipy pandas scikit-learn matplotlib torch
import os, time, math, copy, random, warnings, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.special import jn_zeros, j0, j1
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")
try:
    import wfdb
    HAS_WFDB = True
except ImportError:
    HAS_WFDB = False

CFG = dict(
    SEED=42,
    FS=360,                 # common sampling rate after resampling
    PRE=90, POST=150,       # samples before / after the R-peak (0.25 s / 0.42 s)
    FAST_MODE=True,         # True: quick demo budget. False: full budget
    USE_DETECTED_PEAKS=False,   # False: annotation R-peaks (standard); True: use own detector + label matching
    FANTASIA_PER_GROUP=5,   # records per group (young/old, 2 databases) -> 20 records in total
    FANTASIA_MINUTES=10,    # minutes read per Fantasia record
    FANTASIA_MAX_BEATS=1500,# beats kept per Fantasia record
    MITDB_MAX_BEATS=None,   # None = keep all
    FBSE_M=64, N_WIN=8, N_BANDS=16,
    N_CV_FOLDS=3,           # patient-wise GroupKFold on top of the inter-patient split (0 = skip)
)
if CFG["FAST_MODE"]:
    CFG.update(EPOCHS=12, TUNE_EPOCHS=4, TUNE_TRAIN_N=12000, TUNE_VAL_N=5000, DMRO_POP=6, DMRO_ITERS=4, SYNTH_SECONDS=240)
else:
    CFG.update(EPOCHS=40, TUNE_EPOCHS=8, TUNE_TRAIN_N=40000, TUNE_VAL_N=10000, DMRO_POP=10, DMRO_ITERS=10, SYNTH_SECONDS=900)

FS, PRE, POST = CFG["FS"], CFG["PRE"], CFG["POST"]
BEAT_LEN = PRE + POST
CLASS_NAMES = ["N", "S", "V", "F", "Q"]
N_CLASSES = len(CLASS_NAMES)
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| wfdb:", HAS_WFDB, "| beat length:", BEAT_LEN)

## 2. Data loading (MIT-BIH + Fantasia, with offline synthetic fallback)

In [ ]:
# AAMI mapping of beat annotation symbols
AAMI_MAP = {}
for s in ["N", "L", "R", "e", "j"]: AAMI_MAP[s] = 0      # N : normal / bundle branch / escape
for s in ["A", "a", "J", "S"]:      AAMI_MAP[s] = 1      # S : supraventricular ectopic
for s in ["V", "E"]:                AAMI_MAP[s] = 2      # V : ventricular ectopic
for s in ["F"]:                     AAMI_MAP[s] = 3      # F : fusion
for s in ["/", "f", "Q"]:           AAMI_MAP[s] = 4      # Q : paced / unknown

MITDB_PACED = {"102", "104", "107", "217"}
DS1 = ["101","106","108","109","112","114","115","116","118","119","122","124","201","203","205","207","208","209","215","220","223","230"]
DS2 = ["100","103","105","111","113","117","121","123","200","202","210","212","213","214","219","221","222","228","231","232","233","234"]
FANTASIA_ALL = [f"f{s}{g}{i:02d}" for s in (1, 2) for g in ("y", "o") for i in range(1, 11)]

def read_physionet_record(db_dir, rec, ann_ext, sampto=None):
    r = wfdb.rdrecord(rec, pn_dir=db_dir, sampto=sampto)
    names = [n.strip() for n in r.sig_name]
    ch = next((i for i, n in enumerate(names) if n in ("MLII", "ECG")), 0)
    sig = np.nan_to_num(r.p_signal[:, ch].astype(np.float64))
    ann = wfdb.rdann(rec, ann_ext, pn_dir=db_dir, sampto=sampto)
    keep = [(s, l) for s, l in zip(ann.sample, ann.symbol) if l in AAMI_MAP]
    samples = np.array([k[0] for k in keep], dtype=int)
    labels = np.array([AAMI_MAP[k[1]] for k in keep], dtype=int)
    return dict(sig=sig, fs=int(r.fs), ann_samples=samples, ann_labels=labels)

# ---------------- synthetic fallback (so the notebook runs offline) ----------------
def synth_record(fs, dur_s, class_probs, rng):
    n = int(fs * dur_s); t = np.arange(n) / fs
    hr = rng.uniform(58, 95); rr_mean = 60.0 / hr
    amp = rng.uniform(0.8, 1.4); qw = rng.uniform(0.008, 0.014)
    t_amp = rng.uniform(0.2, 0.4); p_amp = rng.uniform(0.08, 0.2)
    sig = np.zeros(n); peaks, labels = [], []
    def g(t0, w, a):
        lo, hi = max(0, int((t0 - 6 * w) * fs)), min(n, int((t0 + 6 * w) * fs))
        tt = np.arange(lo, hi) / fs
        sig[lo:hi] += a * np.exp(-((tt - t0) ** 2) / (2 * w ** 2))
    pos, pause = rng.uniform(0.4, 0.9), False
    while pos < dur_s - 0.9:
        c = int(rng.choice(N_CLASSES, p=class_probs))
        rr = rr_mean * rng.normal(1, 0.04)
        if c in (1, 2, 3): rr *= rng.uniform(0.55, 0.8)
        if pause: rr *= 1.35; pause = False
        pos += rr
        if pos > dur_s - 0.9: break
        if c == 0:
            g(pos-.16,.025,p_amp); g(pos-.025,qw,-.15*amp); g(pos,qw,amp); g(pos+.025,qw,-.2*amp); g(pos+.25,.045,t_amp)
        elif c == 1:
            g(pos-.12,.02,.3*p_amp); g(pos-.025,qw,-.12*amp); g(pos,qw*1.1,.95*amp); g(pos+.025,qw,-.2*amp); g(pos+.24,.045,t_amp*.9); pause = True
        elif c == 2:
            g(pos,qw*3.5,1.3*amp); g(pos+.06,.03,-.4*amp); g(pos+.28,.06,-.35*amp); pause = True
        elif c == 3:
            g(pos-.14,.025,.6*p_amp); g(pos,qw*2,.9*amp); g(pos+.04,.02,-.15*amp); g(pos+.26,.05,.5*t_amp)
        else:
            sgn = rng.choice([-1, 1]); g(pos, qw*rng.uniform(1,4), sgn*rng.uniform(.4,1.5)*amp); g(pos+.25, .05, rng.uniform(-.4,.4))
        peaks.append(int(pos * fs)); labels.append(c)
    sig += 0.3*np.sin(2*np.pi*rng.uniform(.1,.3)*t + rng.uniform(0,6.28)) + 0.02*np.sin(2*np.pi*50*t) + 0.03*rng.normal(size=n)
    return dict(sig=sig, fs=fs, ann_samples=np.array(peaks), ann_labels=np.array(labels))

def synthetic_records(rng):
    recs = []
    for i in range(44):
        p = rng.dirichlet([40, 2.5, 3.0, 0.8, 1.2]) if i % 5 else np.array([.97, .01, .01, .005, .005])
        r = synth_record(FS, CFG["SYNTH_SECONDS"], p, rng)
        r.update(name=f"synmit_{i:02d}", db="mitdb", split=("ds1" if i < 22 else "ds2")); recs.append(r)
    for i in range(CFG["FANTASIA_PER_GROUP"] * 4):
        r = synth_record(250, CFG["SYNTH_SECONDS"], np.array([1., 0, 0, 0, 0]), rng)
        r.update(name=f"synfan_{i:02d}", db="fantasia", split="fantasia"); recs.append(r)
    return recs

def load_all_records():
    recs = []
    if HAS_WFDB:
        try:
            for rec in DS1 + DS2:
                if rec in MITDB_PACED: continue
                r = read_physionet_record("mitdb", rec, "atr"); r.update(name=rec, db="mitdb", split="ds1" if rec in DS1 else "ds2"); recs.append(r)
            sel = []
            for s in (1, 2):
                for g in ("y", "o"):
                    sel += [f"f{s}{g}{i:02d}" for i in range(1, CFG["FANTASIA_PER_GROUP"] + 1)]
            for rec in sel:
                r = read_physionet_record("fantasia", rec, "ecg", sampto=CFG["FANTASIA_MINUTES"] * 60 * 250)
                r.update(name=rec, db="fantasia", split="fantasia"); recs.append(r)
        except Exception as e:
            print("PhysioNet download failed -> synthetic fallback. Reason:", repr(e)[:150]); recs = []
    if not recs:
        print("USING SYNTHETIC ECG (offline). Results are for pipeline validation only, NOT for the paper.")
        recs = synthetic_records(np.random.default_rng(CFG["SEED"]))
    return recs

RAW_RECORDS = load_all_records()
print(len(RAW_RECORDS), "records loaded:", pd.Series([r["db"] for r in RAW_RECORDS]).value_counts().to_dict())

## 3. Preprocessing: resampling, first-order high-pass IIR, band-pass, scaling

In [ ]:
def resample_sig(x, fs_in, fs_out):
    if fs_in == fs_out: return x
    g = math.gcd(int(fs_in), int(fs_out))
    return signal.resample_poly(x, fs_out // g, fs_in // g)

def highpass_iir_first_order(x, fs, fc=0.5):
    # y[n] = a (y[n-1] + x[n] - x[n-1]),  a = 1 / (1 + 2*pi*fc/fs)
    a = 1.0 / (1.0 + 2 * np.pi * fc / fs)
    return signal.lfilter([a, -a], [1.0, -a], x)

def bandpass(x, fs, lo=0.5, hi=40.0, order=4):
    sos = signal.butter(order, [lo, hi], btype="band", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, x)

def scale_signal(x):
    z = (x - x.mean()) / (x.std() + 1e-8)
    return np.clip(z, -10, 10)

def preprocess_signal(x, fs_in):
    x = resample_sig(x, fs_in, FS)
    x = highpass_iir_first_order(x, FS)
    x = bandpass(x, FS)
    return scale_signal(x)

# ---------------- R-peak detection (Pan-Tompkins style) ----------------
def pan_tompkins(x, fs):
    sos = signal.butter(2, [5, 15], btype="band", fs=fs, output="sos")
    f = signal.sosfiltfilt(sos, x)
    sq = np.gradient(f) ** 2
    w = int(0.15 * fs)
    mwi = np.convolve(sq, np.ones(w) / w, mode="same")
    thr = 0.25 * pd.Series(mwi).rolling(int(4 * fs), center=True, min_periods=1).max().values
    cand, _ = signal.find_peaks(mwi, distance=int(0.25 * fs))
    cand = cand[mwi[cand] > thr[cand]]
    half = int(0.08 * fs); out = []
    for c in cand:                                   # refine to the local R maximum in the filtered ECG
        lo, hi = max(0, c - 2 * half), min(len(x), c + half)
        out.append(lo + int(np.argmax(np.abs(x[lo:hi]))))
    return np.unique(np.array(out, dtype=int))

def match_peaks(det, ref, tol):
    if len(det) == 0 or len(ref) == 0: return 0.0, 0.0
    idx = np.searchsorted(det, ref); tp = 0
    for r, i in zip(ref, idx):
        c = [det[j] for j in (i - 1, i) if 0 <= j < len(det)]
        tp += int(min(abs(np.array(c) - r)) <= tol)
    return tp / len(ref), tp / len(det)           # sensitivity, PPV

# ---------------- segmentation ----------------
def segment_beats(sig, peaks, labels):
    X, Y, RR = [], [], []
    for i in range(1, len(peaks) - 1):
        p = peaks[i]
        if p - PRE < 0 or p + POST > len(sig): continue
        pre, post = (p - peaks[i-1]) / FS, (peaks[i+1] - p) / FS
        k0 = max(0, i - 10); loc = np.mean(np.diff(peaks[k0:i+1])) / FS if i - k0 >= 1 else pre
        X.append(sig[p-PRE:p+POST]); Y.append(labels[i])
        RR.append([pre, post, pre / loc, post / loc])
    return np.array(X, np.float32), np.array(Y, int), np.array(RR, np.float32)

def process_record(r, rng):
    x = preprocess_signal(r["sig"], r["fs"])
    ann = np.round(r["ann_samples"] * FS / r["fs"]).astype(int); lab = r["ann_labels"]
    ok = (ann >= 0) & (ann < len(x)); ann, lab = ann[ok], lab[ok]
    det = pan_tompkins(x, FS)
    sens, ppv = match_peaks(det, ann, int(0.075 * FS))
    if CFG["USE_DETECTED_PEAKS"]:
        peaks, labels = [], []
        for d in det:
            j = np.argmin(np.abs(ann - d))
            if abs(ann[j] - d) <= int(0.15 * FS): peaks.append(d); labels.append(lab[j])
        peaks, labels = np.array(peaks), np.array(labels)
    else:
        peaks, labels = ann, lab
    X, Y, RR = segment_beats(x, peaks, labels)
    cap = CFG["FANTASIA_MAX_BEATS"] if r["db"] == "fantasia" else CFG["MITDB_MAX_BEATS"]
    if cap and len(X) > cap:
        sel = np.sort(rng.choice(len(X), cap, replace=False)); X, Y, RR = X[sel], Y[sel], RR[sel]
    return dict(name=r["name"], db=r["db"], split=r["split"], X=X, y=Y, rr=RR, det_sens=sens, det_ppv=ppv, sig=x, peaks=peaks)

rng_np = np.random.default_rng(CFG["SEED"])
PROCESSED = [process_record(r, rng_np) for r in RAW_RECORDS]
det_df = pd.DataFrame([dict(record=p["name"], db=p["db"], det_sensitivity=p["det_sens"], det_ppv=p["det_ppv"], beats=len(p["y"])) for p in PROCESSED])
print(det_df.groupby("db")[["det_sensitivity", "det_ppv", "beats"]].agg({"det_sensitivity": "mean", "det_ppv": "mean", "beats": "sum"}).round(4))

### 3.1 Visual check: raw vs preprocessed signal and detected R-peaks

In [ ]:
r0 = RAW_RECORDS[0]; p0 = PROCESSED[0]
raw = resample_sig(r0["sig"], r0["fs"], FS)[: 6 * FS]
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(np.arange(len(raw)) / FS, raw, lw=.8); ax[0].set_title(f"Raw ECG (record {p0['name']})")
x = p0["sig"][: 6 * FS]; ax[1].plot(np.arange(len(x)) / FS, x, lw=.8)
det = pan_tompkins(p0["sig"], FS); det = det[det < 6 * FS]
ax[1].plot(det / FS, x[det], "rv", label="detected R-peaks"); ax[1].legend(); ax[1].set_title("Preprocessed ECG"); ax[1].set_xlabel("time (s)")
plt.tight_layout(); plt.savefig(OUT / "preprocessing.png", dpi=150); plt.show()

## 4. Build dataset and patient-independent split

In [ ]:
def concat(procs):
    return dict(
        X=np.concatenate([p["X"] for p in procs]), y=np.concatenate([p["y"] for p in procs]),
        rr=np.concatenate([p["rr"] for p in procs]),
        rec=np.concatenate([[p["name"]] * len(p["y"]) for p in procs]),
        db=np.concatenate([[p["db"]] * len(p["y"]) for p in procs]))

rs = np.random.default_rng(CFG["SEED"])
ds1 = [p for p in PROCESSED if p["split"] == "ds1"]
ds2 = [p for p in PROCESSED if p["split"] == "ds2"]
fan = [p for p in PROCESSED if p["split"] == "fantasia"]
fan_idx = rs.permutation(len(fan)); n_ftr = int(round(0.7 * len(fan)))
train_procs = ds1 + [fan[i] for i in fan_idx[:n_ftr]]
test_procs  = ds2 + [fan[i] for i in fan_idx[n_ftr:]]
val_n = max(3, int(0.15 * len(train_procs))); val_ids = set(rs.choice(len(train_procs), val_n, replace=False))
val_procs = [p for i, p in enumerate(train_procs) if i in val_ids]
train_procs = [p for i, p in enumerate(train_procs) if i not in val_ids]

TR, VA, TE = concat(train_procs), concat(val_procs), concat(test_procs)
assert not (set(TR["rec"]) & set(TE["rec"])) and not (set(TR["rec"]) & set(VA["rec"])), "patient leakage!"
summary = pd.DataFrame({k: np.bincount(d["y"], minlength=N_CLASSES) for k, d in dict(train=TR, val=VA, test=TE).items()}, index=CLASS_NAMES)
print(summary); print({k: len(set(d["rec"])) for k, d in dict(train=TR, val=VA, test=TE).items()}, "records per split")
summary.plot(kind="bar", logy=True, figsize=(7, 3.5), title="Class distribution (log scale)"); plt.tight_layout(); plt.savefig(OUT / "class_distribution.png", dpi=150); plt.show()

## 5. FBSE-based Wasserstein Descriptor (FBSE-WD)

Fourier-Bessel series of a signal $x[n]$, $n=0..N-1$ (zero-order Bessel $J_0$, roots $\lambda_m$ of $J_0$):

$$x[n]=\sum_{m=1}^{M} C_m J_0\!\left(\tfrac{\lambda_m n}{N}\right),\qquad C_m=\frac{2}{N^2 J_1^2(\lambda_m)}\sum_{n} n\,x[n]\,J_0\!\left(\tfrac{\lambda_m n}{N}\right)$$

The (unnormalised) FBSE energy is $E_m\propto (N J_1(\lambda_m) C_m)^2$ and the order $m$ maps to frequency $f_m\approx \lambda_m f_s/(2\pi N)$. This gives a non-stationary-friendly, non-linear time-frequency view with no window leakage.

**Descriptor (50-D) per beat**, all expressed as 1-D Wasserstein distances (earth mover's distance) against a *normal-beat reference* learned on the training set only:

| group | dim | meaning |
|---|---|---|
| global spectral W1 | 1 | W1 between beat FBSE energy distribution and reference |
| window amplitude W1 | 8 | W1 between amplitude distribution of each local window and the reference window |
| window spectral W1 | 8 | W1 between local-window FBSE energy distributions and reference |
| window energy share | 8 | normalised local energy pattern |
| adjacent-window W1 | 7 | distributional change between neighbouring windows |
| FBSE band energies | 16 | log band energies of the global FBSE spectrum |
| centroid + entropy | 2 | spectral summary |

In [ ]:
class FBSE:
    def __init__(self, N, M):
        n = np.arange(N); lam = jn_zeros(0, M); self.J1 = j1(lam); self.lam, self.N, self.M = lam, N, M
        self.A = (2.0 * n[None, :] * j0(lam[:, None] * n[None, :] / N)) / (N ** 2 * self.J1[:, None] ** 2)   # (M, N)
    def coeffs(self, X): return X @ self.A.T
    def energy(self, X): return (self.coeffs(X) * self.N * self.J1) ** 2
    def freqs(self, fs): return self.lam * fs / (2 * np.pi * self.N)

def w1_hist(p, q):          # 1-D Wasserstein-1 between histograms on a unit-length axis
    return np.abs(np.cumsum(p, -1) - np.cumsum(q, -1)).sum(-1) / p.shape[-1]

class FBSEWD:
    EPS = 1e-8
    def __init__(self, M=CFG["FBSE_M"], n_win=CFG["N_WIN"], n_bands=CFG["N_BANDS"]):
        self.M, self.nw, self.nb = M, n_win, n_bands
        self.wl = BEAT_LEN // n_win; self.Mw = max(8, self.wl // 2)
        self.fg, self.fw = FBSE(BEAT_LEN, M), FBSE(self.wl, self.Mw)
        self.bands = np.array_split(np.arange(M), n_bands)
        self.dim = 1 + n_win + n_win + n_win + (n_win - 1) + n_bands + 2
    def _norm(self, E): return E / (E.sum(-1, keepdims=True) + self.EPS)
    def fit(self, Xnormal):
        n = len(Xnormal)
        self.ref_spec = self._norm(self.fg.energy(Xnormal)).mean(0)
        W = Xnormal.reshape(n, self.nw, self.wl)
        self.ref_q = np.sort(W, -1).mean(0)
        self.ref_wspec = self._norm(self.fw.energy(W.reshape(-1, self.wl)).reshape(n, self.nw, self.Mw)).mean(0)
        return self
    def transform(self, X, chunk=20000):
        return np.concatenate([self._t(X[i:i + chunk]) for i in range(0, len(X), chunk)]).astype(np.float32)
    def _t(self, X):
        n = len(X); Pg = self._norm(self.fg.energy(X))
        f1 = w1_hist(Pg, self.ref_spec[None])[:, None]
        W = X.reshape(n, self.nw, self.wl); Wq = np.sort(W, -1)
        f2 = np.abs(Wq - self.ref_q[None]).mean(-1)
        Pw = self._norm(self.fw.energy(W.reshape(-1, self.wl)).reshape(n, self.nw, self.Mw))
        f3 = w1_hist(Pw, self.ref_wspec[None])
        ew = (W ** 2).sum(-1); f4 = ew / (ew.sum(-1, keepdims=True) + self.EPS)
        f5 = np.abs(Wq[:, 1:] - Wq[:, :-1]).mean(-1)
        f6 = np.log(np.stack([Pg[:, b].sum(-1) for b in self.bands], -1) + 1e-6)
        cent = (Pg * np.linspace(0, 1, self.M)).sum(-1, keepdims=True)
        ent = -(Pg * np.log(Pg + self.EPS)).sum(-1, keepdims=True) / np.log(self.M)
        return np.concatenate([f1, f2, f3, f4, f5, f6, cent, ent], 1)

def prepare_features(Xtr, ytr):
    # reference fitted ONLY on training normal beats; scaler fitted ONLY on training features
    wd = FBSEWD().fit(Xtr[ytr == 0] if (ytr == 0).sum() > 50 else Xtr)
    sc = StandardScaler().fit(np.hstack([wd.transform(Xtr), np.zeros((len(Xtr), 4), np.float32)])[:, :wd.dim])
    return wd, sc

class FeatureBuilder:
    def __init__(self, Xtr, RRtr, ytr):
        self.wd = FBSEWD().fit(Xtr[ytr == 0] if (ytr == 0).sum() > 50 else Xtr)
        self.sc = StandardScaler().fit(np.hstack([self.wd.transform(Xtr), RRtr]))
        self.n_wd = self.wd.dim
    def __call__(self, X, RR): return self.sc.transform(np.hstack([self.wd.transform(X), RR])).astype(np.float32)

t0 = time.time(); FB = FeatureBuilder(TR["X"], TR["rr"], TR["y"])
F_TR, F_VA, F_TE = FB(TR["X"], TR["rr"]), FB(VA["X"], VA["rr"]), FB(TE["X"], TE["rr"])
print("descriptor dim (FBSE-WD + 4 RR):", F_TR.shape[1], "| extraction time %.1fs" % (time.time() - t0))

# visual: mean FBSE energy spectrum per class and the first FBSE-WD descriptors
fb = FB.wd.fg; freqs = fb.freqs(FS)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for k in range(N_CLASSES):
    m = TR["y"] == k
    if m.sum() > 5: ax[0].plot(freqs, np.log10(fb.energy(TR["X"][m]).mean(0) + 1e-9), label=CLASS_NAMES[k])
ax[0].set_xlabel("Hz"); ax[0].set_ylabel("log10 FBSE energy"); ax[0].legend(); ax[0].set_title("Mean FBSE spectrum per class")
for k in range(N_CLASSES):
    m = TR["y"] == k
    if m.sum() > 5: ax[1].plot(F_TR[m][:, 1:9].mean(0), marker="o", label=CLASS_NAMES[k])
ax[1].set_xlabel("local window"); ax[1].set_title("Window-wise amplitude-Wasserstein descriptor (standardised)"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / "fbse_wd.png", dpi=150); plt.show()

## 6. R-TCN: R-peak-aware Twisted Convolutional Network

**Twisted convolution layer.** A standard convolution is extended with a position-dependent phase "twist" (inspired by twisted convolution on the Heisenberg group, $ (f\star_\tau g)(t)=\sum_k f(t-k)\,g(k)\,e^{i\tau t k}$). With real/imaginary kernel pair $(W^c, W^s)$ and learnable twist $\tau$:

$$y(t)=\sum_k W^c_k\,x(t+k)\cos\phi_{t,k}+W^s_k\,x(t+k)\sin\phi_{t,k},\qquad \phi_{t,k}=\pi\,\tau\,\tilde t\,k$$

where $\tilde t$ is the time **relative to the R-peak** (R-peak-aware), so the same filter responds differently in the P / QRS / T regions.

**Other parts:** R-peak positional channels (relative time + Gaussian R-window), dilated twisted residual blocks, a **region-interaction attention** block (self-attention between pooled temporal regions, i.e. between temporally separated cardiac segments), R-centred + global pooling, and FBSE-WD / RR descriptor fusion.

In [ ]:
class TwistedConv1d(nn.Module):
    def __init__(self, cin, cout, k=7, dil=1, twist=True, twist_init=0.1, r_frac=PRE / BEAT_LEN):
        super().__init__()
        assert k % 2 == 1
        self.cin, self.cout, self.k, self.dil, self.twist, self.r_frac = cin, cout, k, dil, twist, r_frac
        self.wc = nn.Parameter(torch.empty(cout, cin, k)); nn.init.kaiming_uniform_(self.wc, a=math.sqrt(5))
        if twist:
            self.ws = nn.Parameter(torch.empty(cout, cin, k)); nn.init.kaiming_uniform_(self.ws, a=math.sqrt(5))
            self.alpha = nn.Parameter(torch.tensor(float(twist_init)))
        self.bias = nn.Parameter(torch.zeros(cout))
        self.pad = dil * (k - 1) // 2
        self.register_buffer("tap", torch.arange(k) * dil)
        self.register_buffer("offs", ((torch.arange(k) - k // 2) * dil).float())
    def forward(self, x):
        B, C, T = x.shape
        xp = F.pad(x, (self.pad, self.pad))
        idx = torch.arange(T, device=x.device)[:, None] + self.tap[None, :]          # (T, k)
        xu = xp[:, :, idx]                                                            # (B, C, T, k)
        if self.twist:
            t = (torch.arange(T, device=x.device, dtype=x.dtype) - self.r_frac * T) / (T / 2)   # time relative to R-peak
            ph = self.alpha * math.pi * t[:, None] * self.offs[None, :]               # (T, k)
            xc, xs = xu * torch.cos(ph), xu * torch.sin(ph)
            out = xc.permute(0, 2, 1, 3).reshape(B, T, -1) @ self.wc.reshape(self.cout, -1).T \
                + xs.permute(0, 2, 1, 3).reshape(B, T, -1) @ self.ws.reshape(self.cout, -1).T
        else:
            out = xu.permute(0, 2, 1, 3).reshape(B, T, -1) @ self.wc.reshape(self.cout, -1).T
        return out.transpose(1, 2) + self.bias[None, :, None]

class TwistedBlock(nn.Module):
    def __init__(self, cin, cout, k, dil, drop, twist, twist_init):
        super().__init__()
        self.c1 = TwistedConv1d(cin, cout, k, 1, twist, twist_init); self.b1 = nn.BatchNorm1d(cout)
        self.c2 = TwistedConv1d(cout, cout, k, dil, twist, twist_init); self.b2 = nn.BatchNorm1d(cout)
        self.skip = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity(); self.drop = nn.Dropout(drop)
    def forward(self, x):
        h = self.drop(F.gelu(self.b1(self.c1(x)))); h = self.b2(self.c2(h))
        return F.gelu(h + self.skip(x))

class RegionInteraction(nn.Module):
    # self-attention between pooled temporal regions -> models interaction of temporally separated cardiac segments
    def __init__(self, ch, n_tok=12, heads=4):
        super().__init__()
        self.n_tok = n_tok; self.pos = nn.Parameter(torch.zeros(1, n_tok, ch))
        self.attn = nn.MultiheadAttention(ch, heads, batch_first=True); self.norm = nn.LayerNorm(ch)
        self.gate = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        B, C, T = x.shape
        tok = F.adaptive_avg_pool1d(x, self.n_tok).transpose(1, 2) + self.pos
        a, _ = self.attn(tok, tok, tok, need_weights=False); a = self.norm(tok + a)
        up = F.interpolate(a.transpose(1, 2), size=T, mode="linear", align_corners=False)
        return x + torch.tanh(self.gate) * up

class RTCN(nn.Module):
    def __init__(self, desc_dim, n_classes=N_CLASSES, base_ch=24, kernel=7, dropout=0.3, twist=True, twist_init=0.1):
        super().__init__()
        c = base_ch
        self.stem = TwistedConv1d(3, c, kernel, 1, twist, twist_init); self.bn = nn.BatchNorm1d(c)
        self.b1 = TwistedBlock(c, c, kernel, 1, dropout, twist, twist_init)
        self.b2 = TwistedBlock(c, 2 * c, kernel, 2, dropout, twist, twist_init)
        self.b3 = TwistedBlock(2 * c, 4 * c, kernel, 4, dropout, twist, twist_init)
        self.pool = nn.AvgPool1d(2); self.ri = RegionInteraction(4 * c)
        self.desc = nn.Sequential(nn.Linear(desc_dim, 64), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Sequential(nn.Linear(4 * c * 3 + 64, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, n_classes))
        tt = torch.arange(BEAT_LEN).float() - PRE
        self.register_buffer("pos_ch", torch.stack([tt / (BEAT_LEN / 2), torch.exp(-0.5 * (tt / (0.04 * FS)) ** 2)])[None])
    def forward(self, x, f):
        h = torch.cat([x, self.pos_ch.expand(x.shape[0], -1, -1)], 1)           # (B, 3, T): ECG + R-relative time + R window
        h = F.gelu(self.bn(self.stem(h)))
        h = self.pool(self.b1(h)); h = self.pool(self.b2(h)); h = self.ri(self.b3(h))
        r = int(round(PRE / BEAT_LEN * h.shape[-1]))
        z = torch.cat([h.mean(-1), h.amax(-1), h[:, :, r]], 1)                  # global avg, global max, R-centred feature
        return self.head(torch.cat([z, self.desc(f)], 1))

m = RTCN(F_TR.shape[1]); print("R-TCN parameters: %.1fk" % (sum(p.numel() for p in m.parameters()) / 1e3))
with torch.no_grad(): print(m(torch.randn(4, 1, BEAT_LEN), torch.randn(4, F_TR.shape[1])).shape)

## 7. Training utilities and evaluation metrics

In [ ]:
class BeatDS(Dataset):
    def __init__(self, X, Fe, y):
        self.X, self.F, self.y = torch.tensor(X[:, None, :]), torch.tensor(Fe), torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.F[i], self.y[i]

def class_weights(y, power=0.5):
    cnt = np.bincount(y, minlength=N_CLASSES).astype(float); w = np.ones(N_CLASSES)
    nz = cnt > 0; w[nz] = (cnt[nz].sum() / (nz.sum() * cnt[nz])) ** power
    return w / w[nz].mean()

@torch.no_grad()
def predict_proba(model, X, Fe, bs=2048):
    model.eval().to(DEVICE); out = []
    for i in range(0, len(X), bs):
        xb = torch.tensor(X[i:i+bs, None, :]).to(DEVICE); fb = torch.tensor(Fe[i:i+bs]).to(DEVICE)
        out.append(F.softmax(model(xb, fb), 1).cpu().numpy())
    return np.concatenate(out)

def fit(model, Xtr, Ftr, ytr, Xva, Fva, yva, epochs=20, lr=2e-3, wd=1e-4, ls=0.0, bs=256, patience=6,
        kd=False, kd_w=0.5, kd_T=2.0, verbose=True):
    model.to(DEVICE)
    cw = torch.tensor(class_weights(ytr), dtype=torch.float32, device=DEVICE)
    dl = DataLoader(BeatDS(Xtr, Ftr, ytr), batch_size=bs, shuffle=True, drop_last=len(Xtr) > bs)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(1, epochs * len(dl)))
    teacher = copy.deepcopy(model).eval() if kd else None
    best_f1, best_state, wait, hist = -1, None, 0, []
    for ep in range(epochs):
        model.train(); tl = 0
        for xb, fb, yb in dl:
            xb, fb, yb = xb.to(DEVICE), fb.to(DEVICE), yb.to(DEVICE)
            out = model(xb, fb); loss = F.cross_entropy(out, yb, weight=cw, label_smoothing=ls)
            if kd and ep >= 2:   # self-knowledge distillation from an EMA copy of the model itself
                with torch.no_grad(): tout = teacher(xb, fb)
                loss = loss + kd_w * kd_T ** 2 * F.kl_div(F.log_softmax(out / kd_T, 1), F.softmax(tout / kd_T, 1), reduction="batchmean")
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            if kd:
                with torch.no_grad():
                    for pt, ps in zip(teacher.parameters(), model.parameters()): pt.mul_(0.99).add_(ps.detach(), alpha=0.01)
                    for bt, bs_ in zip(teacher.buffers(), model.buffers()): bt.copy_(bs_)
            tl += loss.item() * len(yb)
        pv = predict_proba(model, Xva, Fva); f1 = f1_score(yva, pv.argmax(1), average="macro", zero_division=0)
        hist.append(dict(epoch=ep + 1, train_loss=tl / len(ytr), val_macro_f1=f1))
        if verbose: print(f"  ep {ep+1:02d} loss {tl/len(ytr):.4f} val macroF1 {f1:.4f}")
        if f1 > best_f1: best_f1, wait, best_state = f1, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return pd.DataFrame(hist), best_f1

def compute_metrics(y, prob):
    K = N_CLASSES; pred = prob.argmax(1); cm = confusion_matrix(y, pred, labels=range(K)); rows = []
    for k in range(K):
        tp = cm[k, k]; fn = cm[k].sum() - tp; fp = cm[:, k].sum() - tp; tn = cm.sum() - tp - fn - fp
        pr = tp / (tp + fp) if tp + fp else np.nan; rc = tp / (tp + fn) if tp + fn else np.nan
        sp = tn / (tn + fp) if tn + fp else np.nan; f1 = 2 * pr * rc / (pr + rc) if (pr + rc) > 0 else (0.0 if tp + fn else np.nan)
        auc = roc_auc_score(y == k, prob[:, k]) if 0 < (y == k).sum() < len(y) else np.nan
        rows.append(dict(cls=CLASS_NAMES[k], precision=pr, recall=rc, sensitivity=rc, specificity=sp, f1=f1, auc=auc, support=int(cm[k].sum())))
    df = pd.DataFrame(rows).set_index("cls")
    yb, sb = (y != 0).astype(int), 1 - prob[:, 0]
    tpb, fnb = ((pred != 0) & (yb == 1)).sum(), ((pred == 0) & (yb == 1)).sum()
    tnb, fpb = ((pred == 0) & (yb == 0)).sum(), ((pred != 0) & (yb == 0)).sum()
    summ = dict(accuracy=(pred == y).mean(), macro_precision=df.precision.mean(), macro_recall=df.recall.mean(), macro_f1=df.f1.mean(),
                macro_specificity=df.specificity.mean(), macro_auc=df.auc.mean(),
                abn_sensitivity=tpb / max(1, tpb + fnb), abn_specificity=tnb / max(1, tnb + fpb),
                abn_auc=roc_auc_score(yb, sb) if 0 < yb.sum() < len(yb) else np.nan)
    return df, summ, cm

## 8. DM-RO: Differential Mutation-based Raindrop Optimization

Raindrop-inspired population search combined with differential evolution operators. Each iteration:

1. **Splash / flow** (raindrop moves): a drop either *splashes* around itself with a radius that shrinks over iterations (exploration) or *flows downhill* towards the lowest (best) drop (exploitation).
2. **Differential mutation**: DE/rand/1 or DE/current-to-best/1 mutant with randomised F, then binomial crossover with the raindrop candidate (adds diversity / escapes local minima).
3. **Greedy selection** per drop.
4. **Evaporation & condensation**: the worst drops evaporate and re-condense around the best drop (or its opposition point).

Fitness = $1-$ validation macro-F1 of the R-TCN trained for a few epochs on a class-capped sub-sample.

In [ ]:
SPACE = [("base_ch", "choice", [16, 24, 32, 48]), ("kernel", "choice", [5, 7, 9]), ("dropout", "float", (0.1, 0.5)),
         ("lr", "log", (5e-4, 5e-3)), ("weight_decay", "log", (1e-6, 1e-3)), ("twist_init", "log", (0.01, 0.5)),
         ("label_smooth", "float", (0.0, 0.1))]
DEFAULT_HP = dict(base_ch=24, kernel=7, dropout=0.3, lr=2e-3, weight_decay=1e-4, twist_init=0.1, label_smooth=0.0)

def decode(x):
    hp = {}
    for v, (name, kind, dom) in zip(np.clip(x, 0, 1 - 1e-9), SPACE):
        if kind == "choice": hp[name] = dom[int(v * len(dom))]
        elif kind == "float": hp[name] = float(dom[0] + v * (dom[1] - dom[0]))
        else: hp[name] = float(np.exp(np.log(dom[0]) + v * (np.log(dom[1]) - np.log(dom[0]))))
    return hp

class DMRO:
    def __init__(self, fitness, dim, pop=8, iters=8, n_evap=2, seed=0, verbose=True):
        assert pop >= 4
        self.f, self.dim, self.pop, self.iters, self.n_evap, self.rng, self.verbose = fitness, dim, pop, iters, n_evap, np.random.default_rng(seed), verbose
    def run(self):
        rng, pop, dim = self.rng, self.pop, self.dim
        X = rng.random((pop, dim)); fit = np.array([self.f(x) for x in X]); hist = [fit.min()]; log = []
        for it in range(self.iters):
            frac = it / max(1, self.iters - 1); radius = 0.35 * (1 - frac) + 0.03; best = X[fit.argmin()].copy()
            for i in range(pop):
                xi = X[i]
                if rng.random() < 0.5 * (1 - frac) + 0.25:      # splash: local exploration with shrinking radius
                    cand = xi + radius * rng.normal(size=dim) * (rng.random(dim) < 0.7)
                else:                                           # flow: slide downhill towards the lowest drop
                    cand = xi + rng.random(dim) * (best - xi) + 0.5 * radius * rng.normal(size=dim)
                F_ = float(np.clip(rng.normal(0.6, 0.2), 0.2, 1.0)); CR = float(np.clip(rng.normal(0.8, 0.1), 0.3, 1.0))
                r1, r2, r3 = rng.choice([j for j in range(pop) if j != i], 3, replace=False)
                mutant = X[r1] + F_ * (X[r2] - X[r3]) if rng.random() < 0.5 else xi + F_ * (best - xi) + F_ * (X[r2] - X[r3])
                mask = rng.random(dim) < CR; mask[rng.integers(dim)] = True
                trial = np.clip(np.where(mask, mutant, cand), 0, 1); ft = self.f(trial)
                if ft <= fit[i]: X[i], fit[i] = trial, ft
            for w in np.argsort(fit)[-self.n_evap:]:             # evaporation -> condensation near the best drop (or its opposite)
                new = best + radius * rng.normal(size=dim)
                if rng.random() < 0.3: new = 1 - X[w]
                new = np.clip(new, 0, 1); fn = self.f(new)
                if fn < fit[w]: X[w], fit[w] = new, fn
            hist.append(fit.min()); log.append(dict(iter=it + 1, best_fitness=fit.min(), mean_fitness=fit.mean()))
            if self.verbose: print(f"DM-RO iter {it+1}/{self.iters}  best fitness {fit.min():.4f}  mean {fit.mean():.4f}  best hp {decode(X[fit.argmin()])}")
        return X[fit.argmin()], float(fit.min()), hist, pd.DataFrame(log)

# quick sanity test of the optimiser on a toy function
_t = DMRO(lambda x: float(((x - 0.3) ** 2).sum()), 5, pop=8, iters=15, seed=1, verbose=False).run()
print("DM-RO toy test (should be near 0):", round(_t[1], 5))

## 9. Run DM-RO hyper-parameter tuning for R-TCN

In [ ]:
def cap_per_class(y, n_total, rng):
    cap = max(50, n_total // N_CLASSES); idx = []
    for k in range(N_CLASSES):
        ik = np.where(y == k)[0]
        idx.append(rng.choice(ik, min(cap, len(ik)), replace=False) if len(ik) else ik)
    return np.sort(np.concatenate(idx))

rt = np.random.default_rng(CFG["SEED"])
tr_i, va_i = cap_per_class(TR["y"], CFG["TUNE_TRAIN_N"], rt), cap_per_class(VA["y"], CFG["TUNE_VAL_N"], rt)
_cache = {}
def fitness(x):
    hp = decode(x); key = tuple(sorted(hp.items()))
    if key in _cache: return _cache[key]
    seed_all(CFG["SEED"])
    model = RTCN(F_TR.shape[1], base_ch=hp["base_ch"], kernel=hp["kernel"], dropout=hp["dropout"], twist_init=hp["twist_init"])
    _, f1 = fit(model, TR["X"][tr_i], F_TR[tr_i], TR["y"][tr_i], VA["X"][va_i], F_VA[va_i], VA["y"][va_i], epochs=CFG["TUNE_EPOCHS"],
                lr=hp["lr"], wd=hp["weight_decay"], ls=hp["label_smooth"], patience=CFG["TUNE_EPOCHS"], verbose=False)
    _cache[key] = 1 - f1; return 1 - f1

t0 = time.time()
opt = DMRO(fitness, len(SPACE), pop=CFG["DMRO_POP"], iters=CFG["DMRO_ITERS"], seed=CFG["SEED"])
x_best, f_best, conv, dmro_log = opt.run()
BEST_HP = decode(x_best)
print("\nBest hyper-parameters:", BEST_HP, "| val macro-F1 %.4f | %.1f min | %d evaluations" % (1 - f_best, (time.time() - t0) / 60, len(_cache)))
plt.figure(figsize=(5.5, 3.5)); plt.plot(1 - np.array(conv), marker="o"); plt.xlabel("iteration"); plt.ylabel("best validation macro-F1")
plt.title("DM-RO convergence"); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(OUT / "dmro_convergence.png", dpi=150); plt.show()
json.dump(BEST_HP, open(OUT / "best_hyperparameters.json", "w"), indent=2)

## 10. Train the final R-TCN (DM-RO tuned) and evaluate on unseen patients

In [ ]:
def build_rtcn(hp, desc_dim, twist=True):
    return RTCN(desc_dim, base_ch=hp["base_ch"], kernel=hp["kernel"], dropout=hp["dropout"], twist=twist, twist_init=hp["twist_init"])

seed_all(CFG["SEED"])
rtcn = build_rtcn(BEST_HP, F_TR.shape[1])
hist_rtcn, _ = fit(rtcn, TR["X"], F_TR, TR["y"], VA["X"], F_VA, VA["y"], epochs=CFG["EPOCHS"], lr=BEST_HP["lr"],
                   wd=BEST_HP["weight_decay"], ls=BEST_HP["label_smooth"], patience=8)
P_rtcn = predict_proba(rtcn, TE["X"], F_TE)
df_rtcn, sum_rtcn, cm_rtcn = compute_metrics(TE["y"], P_rtcn)
print("\nClass-wise metrics, patient-independent test set (R-TCN, DM-RO tuned):"); print(df_rtcn.round(4)); print({k: round(float(v), 4) for k, v in sum_rtcn.items()})
hist_rtcn.plot(x="epoch", y=["train_loss", "val_macro_f1"], figsize=(6, 3.2), title="R-TCN training"); plt.tight_layout(); plt.savefig(OUT / "training_curve.png", dpi=150); plt.show()

## 11. Comparison models (simplified proxies of FADLEC, PathFusion-Net, SKD-AMBRNet)

In [ ]:
class SE(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__(); self.fc = nn.Sequential(nn.Linear(ch, max(4, ch // r)), nn.ReLU(), nn.Linear(max(4, ch // r), ch), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x.mean(-1))[:, :, None]

def cbr(cin, cout, k, s=1): return nn.Sequential(nn.Conv1d(cin, cout, k, s, k // 2), nn.BatchNorm1d(cout), nn.ReLU())

class FADLECProxy(nn.Module):       # feature-attention deep learning ECG classifier: CNN + channel attention + BiGRU + RR features
    def __init__(self, desc_dim, n=N_CLASSES):
        super().__init__()
        self.cnn = nn.Sequential(cbr(1, 32, 7), nn.MaxPool1d(2), cbr(32, 64, 5), nn.MaxPool1d(2), cbr(64, 64, 3)); self.se = SE(64)
        self.gru = nn.GRU(64, 64, batch_first=True, bidirectional=True); self.head = nn.Sequential(nn.Linear(128 + 4, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n))
    def forward(self, x, f):
        h = self.se(self.cnn(x)).transpose(1, 2); o, _ = self.gru(h)
        return self.head(torch.cat([o.mean(1), f[:, -4:]], 1))

class PathFusionProxy(nn.Module):   # parallel multi-scale paths fused by a 1x1 convolution
    def __init__(self, desc_dim, n=N_CLASSES):
        super().__init__()
        self.paths = nn.ModuleList([nn.Sequential(cbr(1, 24, k), cbr(24, 32, k), nn.MaxPool1d(2), cbr(32, 32, k)) for k in (3, 7, 15)])
        self.fuse = cbr(96, 96, 1); self.head = nn.Sequential(nn.Linear(192 + 4, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n))
    def forward(self, x, f):
        h = self.fuse(torch.cat([p(x) for p in self.paths], 1))
        return self.head(torch.cat([h.mean(-1), h.amax(-1), f[:, -4:]], 1))

class SKDAMBRProxy(nn.Module):      # attention multi-branch residual network (trained with self-knowledge distillation)
    def __init__(self, desc_dim, n=N_CLASSES):
        super().__init__()
        self.stem = cbr(1, 32, 7)
        self.b5 = nn.Sequential(cbr(32, 32, 5), cbr(32, 32, 5)); self.b11 = nn.Sequential(cbr(32, 32, 11), cbr(32, 32, 11))
        self.se = SE(64); self.merge = cbr(64, 64, 3); self.pool = nn.MaxPool1d(2); self.att = nn.Conv1d(64, 1, 1)
        self.head = nn.Sequential(nn.Linear(64 + 4, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n))
    def forward(self, x, f):
        s = self.stem(x); h = self.se(torch.cat([self.b5(s) + s, self.b11(s) + s], 1)); h = self.pool(self.merge(h))
        w = F.softmax(self.att(h), -1); return self.head(torch.cat([(h * w).sum(-1), f[:, -4:]], 1))

BASELINES = {"FADLEC [25] (proxy)": (FADLECProxy, False), "PathFusion-Net [26] (proxy)": (PathFusionProxy, False), "SKD-AMBRNet [30] (proxy)": (SKDAMBRProxy, True)}
results = {"CART-ECG (proposed)": (df_rtcn, sum_rtcn, P_rtcn)}; models = {"CART-ECG (proposed)": rtcn}
for name, (cls, kd) in BASELINES.items():
    seed_all(CFG["SEED"]); print("Training", name); mdl = cls(F_TR.shape[1])
    fit(mdl, TR["X"], F_TR, TR["y"], VA["X"], F_VA, VA["y"], epochs=CFG["EPOCHS"], lr=2e-3, wd=1e-4, patience=8, kd=kd, verbose=False)
    P = predict_proba(mdl, TE["X"], F_TE); d, s, _ = compute_metrics(TE["y"], P); results[name] = (d, s, P); models[name] = mdl
    print("  macro-F1 %.4f | accuracy %.4f" % (s["macro_f1"], s["accuracy"]))

## 12. Comparison tables, ROC curves, confusion matrix, latency

In [ ]:
comp = pd.DataFrame({k: v[1] for k, v in results.items()}).T
print("Overall comparison (inter-patient test set):"); print(comp.round(4))
for metric in ["precision", "recall", "f1", "specificity", "auc"]:
    print(f"\nClass-wise {metric}:"); print(pd.DataFrame({k: v[0][metric] for k, v in results.items()}).round(4))
comp.to_csv(OUT / "comparison_overall.csv")
pd.concat({k: v[0] for k, v in results.items()}).to_csv(OUT / "comparison_classwise.csv")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
for k in range(N_CLASSES):                                           # OvR ROC of proposed model
    yk = (TE["y"] == k).astype(int)
    if 0 < yk.sum() < len(yk):
        fpr, tpr, _ = roc_curve(yk, P_rtcn[:, k]); ax[0].plot(fpr, tpr, label=f"{CLASS_NAMES[k]} (AUC {df_rtcn.auc[CLASS_NAMES[k]]:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=.7); ax[0].set_title("CART-ECG one-vs-rest ROC"); ax[0].legend(); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
for name, (d, s, P) in results.items():                              # abnormal-vs-normal ROC for all models
    fpr, tpr, _ = roc_curve((TE["y"] != 0).astype(int), 1 - P[:, 0]); ax[1].plot(fpr, tpr, label=f"{name} ({s['abn_auc']:.3f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=.7); ax[1].set_title("Normal vs abnormal ROC"); ax[1].legend(fontsize=7); ax[1].set_xlabel("FPR")
cmn = cm_rtcn / np.maximum(cm_rtcn.sum(1, keepdims=True), 1); ax[2].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax[2].set_xticks(range(N_CLASSES)); ax[2].set_xticklabels(CLASS_NAMES); ax[2].set_yticks(range(N_CLASSES)); ax[2].set_yticklabels(CLASS_NAMES)
for i in range(N_CLASSES):
    for j in range(N_CLASSES): ax[2].text(j, i, f"{cm_rtcn[i, j]}\n{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > .5 else "black")
ax[2].set_title("CART-ECG confusion matrix"); ax[2].set_xlabel("predicted"); ax[2].set_ylabel("true")
plt.tight_layout(); plt.savefig(OUT / "roc_confusion.png", dpi=150); plt.show()

comp[["accuracy", "macro_precision", "macro_recall", "macro_f1", "macro_specificity"]].plot(kind="bar", figsize=(9, 3.8), rot=15, ylim=(0, 1.02), title="Model comparison")
plt.tight_layout(); plt.savefig(OUT / "comparison_bar.png", dpi=150); plt.show()

# ---- computational latency (single-beat inference, batch size 1; CART-ECG includes FBSE-WD extraction) ----
def latency_ms(fn, n=100):
    for _ in range(5): fn(0)
    t0 = time.perf_counter()
    for i in range(n): fn(i)
    return (time.perf_counter() - t0) / n * 1000
lat = {}
for name, mdl in models.items():
    mdl.eval().to(DEVICE)
    def run(i, mdl=mdl, name=name):
        j = i % len(TE["X"]); f = F_TE[j:j+1]
        if name.startswith("CART"): f = FB(TE["X"][j:j+1], TE["rr"][j:j+1])     # descriptor extraction counted
        with torch.no_grad(): mdl(torch.tensor(TE["X"][j:j+1, None, :]).to(DEVICE), torch.tensor(f).to(DEVICE))
    lat[name] = dict(latency_ms_per_beat=latency_ms(run), params_k=sum(p.numel() for p in mdl.parameters()) / 1e3)
lat_df = pd.DataFrame(lat).T; print(lat_df.round(3)); lat_df.to_csv(OUT / "latency.csv")

## 13. Patient-independent testing (per-record results + patient-wise cross-validation)

In [ ]:
pred = P_rtcn.argmax(1); rows = []
for rec in np.unique(TE["rec"]):
    m = TE["rec"] == rec
    rows.append(dict(record=rec, db=TE["db"][m][0], beats=int(m.sum()), abnormal_beats=int((TE["y"][m] != 0).sum()),
                     accuracy=(pred[m] == TE["y"][m]).mean(), macro_f1=f1_score(TE["y"][m], pred[m], average="macro", zero_division=0)))
per_rec = pd.DataFrame(rows).set_index("record"); print(per_rec.round(4)); print("\nper-database accuracy:"); print(per_rec.groupby("db").accuracy.mean().round(4))
per_rec.to_csv(OUT / "per_record_results.csv")

def patient_cv(n_folds):
    ALL = dict(X=np.concatenate([TR["X"], VA["X"], TE["X"]]), y=np.concatenate([TR["y"], VA["y"], TE["y"]]),
               rr=np.concatenate([TR["rr"], VA["rr"], TE["rr"]]), rec=np.concatenate([TR["rec"], VA["rec"], TE["rec"]]))
    out = []
    for fold, (a, b) in enumerate(GroupKFold(n_splits=n_folds).split(ALL["X"], ALL["y"], ALL["rec"])):
        recs = np.unique(ALL["rec"][a]); vr = np.random.default_rng(fold).choice(recs, max(2, len(recs) // 8), replace=False)
        vm = np.isin(ALL["rec"][a], vr); ia, iv = a[~vm], a[vm]
        fb = FeatureBuilder(ALL["X"][ia], ALL["rr"][ia], ALL["y"][ia])
        Fa, Fv, Fb = fb(ALL["X"][ia], ALL["rr"][ia]), fb(ALL["X"][iv], ALL["rr"][iv]), fb(ALL["X"][b], ALL["rr"][b])
        seed_all(fold); mdl = build_rtcn(BEST_HP, Fa.shape[1])
        fit(mdl, ALL["X"][ia], Fa, ALL["y"][ia], ALL["X"][iv], Fv, ALL["y"][iv], epochs=CFG["EPOCHS"], lr=BEST_HP["lr"], wd=BEST_HP["weight_decay"],
            ls=BEST_HP["label_smooth"], patience=6, verbose=False)
        _, s, _ = compute_metrics(ALL["y"][b], predict_proba(mdl, ALL["X"][b], Fb)); s["fold"] = fold + 1; out.append(s)
        print(f"fold {fold+1}: acc {s['accuracy']:.4f} macroF1 {s['macro_f1']:.4f} abnormal AUC {s['abn_auc']:.4f}")
    return pd.DataFrame(out).set_index("fold")
if CFG["N_CV_FOLDS"] >= 2:
    cv_df = patient_cv(CFG["N_CV_FOLDS"]); print("\nPatient-wise CV (mean +- std):"); print(pd.DataFrame({"mean": cv_df.mean(), "std": cv_df.std()}).round(4)); cv_df.to_csv(OUT / "patient_cv.csv")

## 14. Ablation study

In [ ]:
def run_variant(name, build, Ftr, Fva, Fte, hp):
    seed_all(CFG["SEED"]); mdl = build()
    fit(mdl, TR["X"], Ftr, TR["y"], VA["X"], Fva, VA["y"], epochs=CFG["EPOCHS"], lr=hp["lr"], wd=hp["weight_decay"], ls=hp["label_smooth"], patience=8, verbose=False)
    P = predict_proba(mdl, TE["X"], Fte); _, s, _ = compute_metrics(TE["y"], P); s["variant"] = name; return s

def drop_wd(Fx): Fz = Fx.copy(); Fz[:, :FB.n_wd] = 0; return Fz          # remove FBSE-WD (keep RR features)
d = F_TR.shape[1]; abl = [dict(CART_ECG_full=sum_rtcn, variant="Full CART-ECG (DM-RO tuned)")]
abl = [dict(sum_rtcn, variant="Full CART-ECG (DM-RO tuned)")]
abl.append(run_variant("w/o FBSE-WD", lambda: build_rtcn(BEST_HP, d), drop_wd(F_TR), drop_wd(F_VA), drop_wd(F_TE), BEST_HP))
abl.append(run_variant("w/o twisted convolution (plain conv)", lambda: build_rtcn(BEST_HP, d, twist=False), F_TR, F_VA, F_TE, BEST_HP))
abl.append(run_variant("w/o DM-RO (default hyper-parameters)", lambda: build_rtcn(DEFAULT_HP, d), F_TR, F_VA, F_TE, DEFAULT_HP))
abl_df = pd.DataFrame(abl).set_index("variant")[["accuracy", "macro_precision", "macro_recall", "macro_f1", "macro_specificity", "macro_auc", "abn_auc"]]
print(abl_df.round(4)); abl_df.to_csv(OUT / "ablation.csv")

## 15. Interpretability: gradient saliency and FBSE-WD group importance

In [ ]:
rtcn.eval().to(DEVICE)
fig, ax = plt.subplots(1, N_CLASSES, figsize=(3.3 * N_CLASSES, 3), sharey=False); t_ax = (np.arange(BEAT_LEN) - PRE) / FS * 1000
for k in range(N_CLASSES):
    idx = np.where(TE["y"] == k)[0][:200]
    if len(idx) == 0: ax[k].set_title(f"{CLASS_NAMES[k]} (none)"); continue
    xb = torch.tensor(TE["X"][idx, None, :], requires_grad=True, device=DEVICE); fb = torch.tensor(F_TE[idx]).to(DEVICE)
    rtcn(xb, fb)[:, k].sum().backward(); sal = (xb.grad * xb).abs().mean(0)[0].detach().cpu().numpy(); sal /= sal.max() + 1e-9
    ax[k].plot(t_ax, TE["X"][idx].mean(0), "k", lw=1.2); ax[k].fill_between(t_ax, TE["X"][idx].mean(0).min(), TE["X"][idx].mean(0).max(), where=sal > 0.5, alpha=.25, color="red")
    ax[k].plot(t_ax, sal * (TE["X"][idx].mean(0).max() - TE["X"][idx].mean(0).min()) + TE["X"][idx].mean(0).min(), "r", lw=.8, alpha=.7); ax[k].set_title(f"class {CLASS_NAMES[k]}"); ax[k].set_xlabel("ms from R-peak")
plt.suptitle("Mean beat (black) and gradient x input saliency (red)"); plt.tight_layout(); plt.savefig(OUT / "saliency.png", dpi=150); plt.show()

groups = {"global spectral W1": (0, 1), "window amplitude W1": (1, 9), "window spectral W1": (9, 17), "window energy share": (17, 25),
          "adjacent-window W1": (25, 32), "FBSE band energies": (32, 48), "centroid+entropy": (48, 50), "RR features": (50, 54)}
base = f1_score(TE["y"], P_rtcn.argmax(1), average="macro", zero_division=0); rng_p = np.random.default_rng(0); imp = {}
for g, (a, b) in groups.items():
    drops = []
    for _ in range(3):
        Fp = F_TE.copy(); Fp[:, a:b] = Fp[rng_p.permutation(len(Fp))][:, a:b]
        drops.append(base - f1_score(TE["y"], predict_proba(rtcn, TE["X"], Fp).argmax(1), average="macro", zero_division=0))
    imp[g] = np.mean(drops)
pd.Series(imp).sort_values().plot(kind="barh", figsize=(6, 3.3), title="Permutation importance (drop in macro-F1)"); plt.tight_layout(); plt.savefig(OUT / "descriptor_importance.png", dpi=150); plt.show()

## 16. Save artefacts

In [ ]:
torch.save(dict(model=rtcn.state_dict(), hp=BEST_HP, desc_dim=F_TR.shape[1]), OUT / "cart_ecg_rtcn.pt")
import pickle; pickle.dump(dict(fbse_wd=FB.wd, scaler=FB.sc), open(OUT / "feature_pipeline.pkl", "wb"))
print("Saved to", OUT.resolve(), "->", sorted(p.name for p in OUT.iterdir()))
print("\nREMINDER: if the first cells printed 'USING SYNTHETIC ECG', re-run with internet access (pip install wfdb) to obtain real MIT-BIH / Fantasia results.")